# 09: Cross-Validation

Every result in notebooks 04 and 08 comes from one train/validation/test split. Notebook
08 showed those numbers move around a lot between runs, which raises a fair question
about the headline figure: was the single-split score the model's real performance, or a
good split?

This notebook answers that by evaluating on five splits instead of one and reporting the
mean and standard deviation. It was the first item on my future-work list, so running
it turns a recommendation into a result.

No new data is needed. Five-fold cross-validation reuses the same 2,200 reviews five
ways: each fold holds out a fifth for testing and trains on the rest, so every review is
tested exactly once and no review is ever trained and tested on in the same fold.

Early stopping still needs a validation set, so within each fold's training portion I
hold out a further 10% for that. The test fold is never touched during training.

**Runtime:** five BERT trainings, roughly an hour on Apple Silicon. The SVM baseline is
included too since it costs seconds and shows whether BERT's advantage holds fold to fold.

Single-split figures to compare against: BERT 0.6754, SVM 0.6596.

## Note on the single-split constant

`SINGLE_BERT_F1` now reads **0.6754**, the delivered checkpoint's test macro F1 after the
focal-loss correction, as re-scored from `models/bert_category`.

The printed output below was produced with the *previous* constant, **0.7079** (the
pre-correction checkpoint's score), still in place, so one line of that output quotes 0.7079.
The cross-validation results are unaffected; the constant is only used for the printed
comparison against the single split, not in the folds themselves. Against the corrected
figures the single split of 0.6754 sits just below the cross-validated mean of 0.6872, which
is the point: a single split is a draw, not an estimate.

## Setup

In [1]:
import json
import warnings

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.svm import LinearSVC
from sklearn.utils.class_weight import compute_class_weight
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import BertForSequenceClassification, BertTokenizer, get_linear_schedule_with_warmup

warnings.filterwarnings('ignore')

NAMES = ['Bug Report', 'Feature Request', 'UX Feedback', 'Positive Praise']
MAX_LEN, BATCH_SIZE, LR, MAX_EPOCHS, PATIENCE, SEED = 128, 16, 2e-5, 6, 2, 42
N_FOLDS = 5

# single-split results, for comparison
SINGLE_BERT_F1, SINGLE_SVM_F1 = 0.6754, 0.6596

if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')
print('Device:', device)

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: mps


In [2]:
# recombine the original splits: cross-validation makes its own
full = pd.concat([
    pd.read_csv('../data/processed/train.csv'),
    pd.read_csv('../data/processed/val.csv'),
    pd.read_csv('../data/processed/test.csv'),
], ignore_index=True)

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

print(f'{len(full)} reviews total')
print(full['category_name'].value_counts().to_string())

2200 reviews total
category_name
Positive Praise    833
Bug Report         743
UX Feedback        439
Feature Request    185


### Helpers

Same training setup as notebook 04, so the only thing changing is how the data is split.

In [3]:
class ReviewDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts, self.labels = list(texts), list(labels)

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, i):
        enc = tokenizer(str(self.texts[i]), max_length=MAX_LEN, padding='max_length',
                        truncation=True, return_tensors='pt')
        return {'input_ids': enc['input_ids'].squeeze(),
                'attention_mask': enc['attention_mask'].squeeze(),
                'label': torch.tensor(self.labels[i], dtype=torch.long)}


def loader(df, shuffle=False):
    return DataLoader(ReviewDataset(df['clean_text'], df['category_label']),
                      batch_size=BATCH_SIZE, shuffle=shuffle)


class FocalLoss(nn.Module):
    def __init__(self, weight=None, gamma=2.0):
        super().__init__()
        self.weight, self.gamma = weight, gamma

    def forward(self, logits, target):
        ce = F.cross_entropy(logits, target, reduction='none')   # unweighted -> true p_t
        focal = ((1 - torch.exp(-ce)) ** self.gamma) * ce
        if self.weight is not None:
            focal = self.weight[target] * focal   # class weight as the focal alpha
        return focal.mean()


@torch.no_grad()
def predict(model, dl):
    model.eval()
    L, Y = [], []
    for b in dl:
        out = model(input_ids=b['input_ids'].to(device),
                    attention_mask=b['attention_mask'].to(device))
        L.append(out.logits.cpu().numpy()); Y.append(b['label'].numpy())
    return np.concatenate(L).argmax(1), np.concatenate(Y)

In [4]:
def train_fold(tr, va, tag):
    torch.manual_seed(SEED); np.random.seed(SEED)
    model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=4).to(device)
    tl, vl = loader(tr, shuffle=True), loader(va)

    no_decay = ['bias', 'LayerNorm.weight']
    grouped = [
        {'params': [p for n, p in model.named_parameters() if not any(d in n for d in no_decay)],
         'weight_decay': 0.01},
        {'params': [p for n, p in model.named_parameters() if any(d in n for d in no_decay)],
         'weight_decay': 0.0},
    ]
    opt = AdamW(grouped, lr=LR, eps=1e-8)
    total = len(tl) * MAX_EPOCHS
    sched = get_linear_schedule_with_warmup(opt, int(0.1 * total), total)

    cw = compute_class_weight('balanced', classes=np.arange(4), y=tr['category_label'].values)
    loss_fn = FocalLoss(torch.tensor(cw, dtype=torch.float).to(device), 2.0)

    import copy
    best, state, patience = -1, None, PATIENCE
    for ep in range(MAX_EPOCHS):
        model.train()
        for b in tqdm(tl, desc=f'{tag} ep{ep+1}', leave=False):
            opt.zero_grad()
            out = model(input_ids=b['input_ids'].to(device),
                        attention_mask=b['attention_mask'].to(device))
            loss_fn(out.logits, b['label'].to(device)).backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); sched.step()

        p, y = predict(model, vl)
        v = f1_score(y, p, average='macro', zero_division=0)
        print(f'    epoch {ep+1}: inner-val macro F1 = {v:.4f}')
        if v > best:
            best, state, patience = v, copy.deepcopy(model.state_dict()), PATIENCE
        else:
            patience -= 1
            if patience == 0:
                print(f'    early stop at epoch {ep+1}')
                break

    model.load_state_dict(state)
    return model

---
## Five-fold cross-validation

Stratified so each fold keeps the same class proportions as the full dataset, which
matters here because Feature Request is only 8% of the data.

In [5]:
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
y_all = full['category_label'].values

bert_f1, bert_acc, bert_per_class = [], [], []
svm_f1, svm_acc = [], []

for k, (tr_idx, te_idx) in enumerate(skf.split(np.zeros(len(y_all)), y_all), 1):
    print(f'\n===== FOLD {k}/{N_FOLDS} =====')
    train_part, test_part = full.iloc[tr_idx], full.iloc[te_idx]

    # inner split for early stopping; the test fold stays untouched
    tr, va = train_test_split(train_part, test_size=0.1, random_state=SEED,
                              stratify=train_part['category_label'])
    print(f'  train {len(tr)} | inner-val {len(va)} | test {len(test_part)}')

    # --- SVM baseline (seconds) ---
    vec = TfidfVectorizer(max_features=10000, ngram_range=(1, 2))
    Xtr = vec.fit_transform(train_part['clean_text'].astype(str))
    Xte = vec.transform(test_part['clean_text'].astype(str))
    svm = LinearSVC(class_weight='balanced', random_state=SEED)
    svm.fit(Xtr, train_part['category_label'])
    sp = svm.predict(Xte)
    svm_f1.append(f1_score(test_part['category_label'], sp, average='macro', zero_division=0))
    svm_acc.append((sp == test_part['category_label'].values).mean())
    print(f'  SVM  macro F1 = {svm_f1[-1]:.4f}')

    # --- BERT ---
    model = train_fold(tr, va, f'fold{k}')
    p, y = predict(model, loader(test_part))
    bert_f1.append(f1_score(y, p, average='macro', zero_division=0))
    bert_acc.append((p == y).mean())
    rep = classification_report(y, p, target_names=NAMES, output_dict=True, zero_division=0)
    bert_per_class.append({n: rep[n]['f1-score'] for n in NAMES})
    print(f'  BERT macro F1 = {bert_f1[-1]:.4f}')

    del model
    if device.type == 'mps':
        torch.mps.empty_cache()


===== FOLD 1/5 =====
  train 1584 | inner-val 176 | test 440
  SVM  macro F1 = 0.6731


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7782.87it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoi

    epoch 1: inner-val macro F1 = 0.4579


    epoch 2: inner-val macro F1 = 0.6736


    epoch 3: inner-val macro F1 = 0.6551


...

    epoch 2: inner-val macro F1 = 0.6373


    epoch 3: inner-val macro F1 = 0.6558


    epoch 4: inner-val macro F1 = 0.6928


    epoch 5: inner-val macro F1 = 0.6937


    epoch 6: inner-val macro F1 = 0.6898
  BERT macro F1 = 0.6931


---
## Results

In [6]:
b, s = np.array(bert_f1), np.array(svm_f1)

print('Per-fold macro F1')
print(f"  {'fold':6s} {'BERT':>8s} {'SVM':>8s}")
for i, (x, y) in enumerate(zip(bert_f1, svm_f1), 1):
    print(f'  {i:<6d} {x:8.4f} {y:8.4f}')

print(f'\nBERT  {b.mean():.4f} +/- {b.std():.4f}   (range {b.min():.4f} to {b.max():.4f})')
print(f'SVM   {s.mean():.4f} +/- {s.std():.4f}   (range {s.min():.4f} to {s.max():.4f})')
print(f'\nSingle-split figures were BERT {SINGLE_BERT_F1}, SVM {SINGLE_SVM_F1}')
print(f'  BERT: single split sits {(SINGLE_BERT_F1 - b.mean()) / b.std():+.1f} standard deviations from the mean')
print(f'  BERT beat SVM in {int((b > s).sum())} of {N_FOLDS} folds')

Per-fold macro F1
  fold       BERT      SVM
  1        0.6916   0.6731
  2        0.6340   0.6645
  3        0.7281   0.6851
  4        0.6891   0.6806
  5        0.6931   0.6598

BERT  0.6872 +/- 0.0302   (range 0.6340 to 0.7281)
SVM   0.6726 +/- 0.0095   (range 0.6598 to 0.6851)

Single-split figures were BERT 0.7079, SVM 0.6596
  BERT: single split sits +0.7 standard deviations from the mean
  BERT beat SVM in 4 of 5 folds


In [7]:
pc = pd.DataFrame(bert_per_class)
summary = pd.DataFrame({'mean F1': pc.mean().round(4), 'std': pc.std().round(4),
                        'min': pc.min().round(4), 'max': pc.max().round(4)})
print('BERT per-class F1 across folds')
print(summary.to_string())

results = {
    'n_folds': N_FOLDS,
    'n_reviews': len(full),
    'note': ('Stratified 5-fold over all 2,200 reviews. Within each fold, 10% of the '
             'training portion is held out for early stopping; the test fold is never '
             'seen during training.'),
    'bert': {'per_fold_macro_f1': [round(x, 4) for x in bert_f1],
             'mean_macro_f1': round(float(b.mean()), 4),
             'std_macro_f1': round(float(b.std()), 4),
             'mean_accuracy': round(float(np.mean(bert_acc)), 4),
             'per_class_mean_f1': summary['mean F1'].to_dict(),
             'per_class_std_f1': summary['std'].to_dict()},
    'svm': {'per_fold_macro_f1': [round(x, 4) for x in svm_f1],
            'mean_macro_f1': round(float(s.mean()), 4),
            'std_macro_f1': round(float(s.std()), 4),
            'mean_accuracy': round(float(np.mean(svm_acc)), 4)},
    'single_split_comparison': {'bert_macro_f1': SINGLE_BERT_F1, 'svm_macro_f1': SINGLE_SVM_F1},
}

with open('../reports/cross_validation_results.json', 'w') as f:
    json.dump(results, f, indent=2)
print('\nSaved ../reports/cross_validation_results.json')

BERT per-class F1 across folds
                 mean F1     std     min     max
Bug Report        0.7388  0.0224  0.7216  0.7766
Feature Request   0.5431  0.0803  0.4058  0.6154
UX Feedback       0.6044  0.0579  0.5093  0.6634
Positive Praise   0.8625  0.0181  0.8348  0.8829

Saved ../reports/cross_validation_results.json


## What to read from this

Three things, and here is how they came out.

First, where the single-split figure sits. The delivered single split of 0.6754 sits just
below the cross-validated mean of 0.6872, and the pre-correction checkpoint's 0.7079 sat
above it, so the same procedure produced both a high and a low draw. That supports the
argument that a single split is a fortunate or unfortunate draw rather than a
stable estimate.

Second, whether BERT beats SVM. With the corrected loss BERT wins four folds of five and
leads on the mean, 0.6872 to 0.6726, but by less than its own standard deviation, so the two
are statistically indistinguishable. Before the loss fix the SVM led and BERT won only one
fold, so a one-line change flipped the ranking, which is the sharpest evidence here that
the comparison is not measurable at this dataset size.

Third, which class is actually weakest. Under cross-validation Feature Request is the
weakest class at 0.54, with UX Feedback near 0.60; the loss fix rebalanced Feature Request,
but UX Feedback stays limited by its ambiguous boundary, consistent with the augmentation
result in notebook 07 and the label audit in notebook 08.

The mean and standard deviation are a more honest headline than one point estimate, even
though the mean, 0.6872, is a shade under the single split's fortunate draws.